# 03 - Full Agent Workflow

This notebook demonstrates the complete Smart Home Agent using LangGraph.

## Learning Objectives

1. Understand LangGraph workflow structure
2. See how nodes process state step-by-step
3. Observe GraphRAG + LLM integration
4. Debug and trace agent reasoning

## Prerequisites

- Completed notebooks 01 and 02
- Neo4j running with seed data
- OpenAI API key configured in `.env`

## Setup

In [1]:
import sys
sys.path.insert(0, '..')

import os
import json
from dotenv import load_dotenv
load_dotenv('../.env')

# Verify API key is set
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"✓ OpenAI API Key: {api_key[:8]}...{api_key[-4:]}")
else:
    print("✗ OpenAI API Key not found!")
    print("  Set OPENAI_API_KEY in ../.env file")

✓ OpenAI API Key: sk-proj-...KpYA


In [2]:
# Import agent components
from src.agent import SmartHomeAgent, create_initial_state, state_summary
from src.agent.workflow import create_workflow, visualize_workflow
from src.graph.retriever import SmartHomeRetriever

print("✓ Imports successful!")

✓ Imports successful!


## 1. Understanding the Workflow

Let's visualize the agent's workflow structure.

In [3]:
# Print the workflow diagram
print(visualize_workflow())


```mermaid
graph TD
    START([Start]) --> parse[Parse Intent]
    parse --> retrieve[Retrieve Context]
    retrieve --> check{Check Sufficiency}

    check -->|Needs Clarification| ask[Ask Clarification]
    check -->|Sufficient| plan[Generate Plan]

    ask --> END1([End])

    plan --> counter[Increment Counter]
    counter --> validate{Validate Plan}

    validate -->|Valid| response[Generate Response]
    validate -->|Invalid & Retry| plan

    response --> END2([End])

    style START fill:#90EE90
    style END1 fill:#FFB6C1
    style END2 fill:#FFB6C1
    style check fill:#FFE4B5
    style validate fill:#FFE4B5
```



### Workflow Nodes Explained

| Node | Purpose | Input | Output |
|------|---------|-------|--------|
| `parse_intent` | Extract structured intent from text | user_input | parsed_intent |
| `retrieve_context` | Query Neo4j via GraphRAG | parsed_intent | retrieved_context |
| `check_sufficiency` | Decide if we have enough info | retrieved_context | needs_clarification |
| `generate_plan` | LLM creates action plan | intent + context | action_plan |
| `validate_plan` | Check plan validity | action_plan | validation_result |
| `generate_response` | Create user-friendly response | action_plan | final_response |

In [4]:
# Create the workflow and inspect it
workflow = create_workflow()
compiled = workflow.compile()

# Get graph info
graph = compiled.get_graph()
print("Workflow Nodes:")
for node in graph.nodes:
    print(f"  - {node}")

Workflow Nodes:
  - __start__
  - parse_intent
  - retrieve_context
  - check_sufficiency
  - ask_clarification
  - generate_plan
  - validate_plan
  - generate_response
  - increment_counter
  - __end__


## 2. Running the Agent (Basic)

Let's run the agent with a simple request.

In [5]:
# Create the agent
agent = SmartHomeAgent(debug=False)

# Simple request
response = agent.run("Turn on the living room lights")

print("Request: Turn on the living room lights")
print("\nResponse:")
print(response)

Request: Turn on the living room lights

Response:
Sure thing! I’m turning on the ceiling light and the floor lamp in the living room to brighten things up. You’ll have plenty of light in no time! If you need anything else, just let me know!


## 3. Running with Trace

Let's see the reasoning steps.

In [6]:
# Run with trace
response, trace = agent.run_with_trace("Make the living room cozy for movie night")

print("Request: Make the living room cozy for movie night")
print("\n📋 Reasoning Trace:")
print("-" * 50)
for i, step in enumerate(trace, 1):
    print(f"{i}. {step}")

print("\n💬 Response:")
print("-" * 50)
print(response)

Request: Make the living room cozy for movie night

📋 Reasoning Trace:
--------------------------------------------------
1. Parsed intent: goal='create cozy atmosphere', rooms=['living room']
2. Retrieved context using strategies: ['room_context', 'scene_lookup']
3. Context is sufficient to generate plan
4. Generated plan with 4 actions
5. Validation: passed
6. Generated user response

💬 Response:
--------------------------------------------------
I'll make the living room all cozy for your movie night! I'm closing the blinds to block out any glare, dimming the lights for that perfect ambiance, and turning on the TV. You're all set for a great evening! Enjoy the show! 🍿


## 4. Inspecting Full State

Let's look at the complete state after processing.

In [7]:
# Get full state
final_state = agent.get_full_state("Set up the bedroom for sleep")

# Print summary
print(state_summary(final_state))

AGENT STATE SUMMARY

📝 User Input: Set up the bedroom for sleep

🎯 Parsed Intent:
   Goal: set up for sleep
   Rooms: ['bedroom']
   Mood: None

📚 Retrieved Context: 653 characters

📋 Action Plan: 3 actions
   - Bedside Lamp: color
   - Bedside Lamp: dim
   - Bedroom Light: dim

✔️  Validation: ✅ Valid

🔄 Iteration: 1

💬 Final Response: I’m setting up your bedroom for a restful night’s sleep! I’ve dimmed the bedside lamp to 20% and swi...



In [8]:
# Inspect individual state components
print("\n🎯 Parsed Intent:")
print(json.dumps(final_state.get('parsed_intent', {}), indent=2))


🎯 Parsed Intent:
{
  "goal": "set up for sleep",
  "rooms": [
    "bedroom"
  ],
  "actions": null,
  "mood": null,
  "scene_hint": null,
  "constraints": null
}


In [9]:
# Inspect action plan
print("\n📋 Action Plan:")
print(json.dumps(final_state.get('action_plan', {}), indent=2))


📋 Action Plan:
{
  "reasoning": "To create a calming environment for sleep, I will dim the lights and set a warm color for the bedside lamp, following the 'Bedtime' scene for optimal relaxation.",
  "actions": [
    {
      "device_id": "bedside_lamp_01",
      "device_name": "Bedside Lamp",
      "capability": "color",
      "value": "warm orange",
      "reason": "A warm color promotes relaxation and prepares the body for sleep."
    },
    {
      "device_id": "bedside_lamp_01",
      "device_name": "Bedside Lamp",
      "capability": "dim",
      "value": 20,
      "reason": "Dimming the light reduces brightness, creating a cozy atmosphere for sleep."
    },
    {
      "device_id": "bedroom_light_01",
      "device_name": "Bedroom Light",
      "capability": "dim",
      "value": 10,
      "reason": "Lowering the brightness of the bedroom light further enhances the sleep environment."
    }
  ]
}


In [10]:
# Inspect retrieved context
context = final_state.get('retrieved_context', '')
print("\n📚 Retrieved Context (first 1000 chars):")
print("-" * 50)
print(context[:1000])
if len(context) > 1000:
    print(f"\n... ({len(context) - 1000} more characters)")


📚 Retrieved Context (first 1000 chars):
--------------------------------------------------
# Smart Home Context

## Room: Master Bedroom
- Type: bedroom
- Floor: First Floor
- Adjacent to: Home Office, Bathroom

### Devices (3 total)
- **Bedroom Speaker** (speaker) ✓
  - Capabilities: announce, play_music, volume, power
- **Bedside Lamp** (light) ✓
  - Capabilities: color, dim, power
- **Bedroom Light** (light) ✓
  - Capabilities: color, dim, power

### Available Scenes
- **Bedtime**: Relaxing settings for sleep preparation
  - Actions: Bedside Lamp: warm orange, dim; Bedroom Light: dim to 10%
- **Good Morning**: Energizing settings to start the day
  - Actions: Bedroom Speaker: play morning playlist; Bedroom Light: bright white light


## 5. Streaming Execution

Watch the agent process step by step.

In [11]:
# Stream execution
print("Streaming: 'Dim the lights in the office'")
print("=" * 50)

for event in agent.run_streaming("Dim the lights in the office"):
    # Each event is a dict with node name as key
    for node_name, updates in event.items():
        print(f"\n📍 Node: {node_name}")
        
        # Show key updates
        if 'parsed_intent' in updates:
            goal = updates['parsed_intent'].get('goal', 'N/A')
            print(f"   → Parsed goal: {goal}")
        
        if 'retrieved_context' in updates:
            ctx_len = len(updates['retrieved_context'])
            print(f"   → Retrieved {ctx_len} chars of context")
        
        if 'action_plan' in updates:
            actions = updates['action_plan'].get('actions', [])
            print(f"   → Generated {len(actions)} actions")
        
        if 'final_response' in updates:
            print(f"   → Response: {updates['final_response'][:100]}...")

Streaming: 'Dim the lights in the office'

📍 Node: parse_intent
   → Parsed goal: None

📍 Node: retrieve_context
   → Retrieved 378 chars of context

📍 Node: check_sufficiency

📍 Node: generate_plan
   → Generated 1 actions

📍 Node: increment_counter

📍 Node: validate_plan

📍 Node: generate_response
   → Response: I'm dimming the desk lamp in your office to 30% to create a softer and more comfortable atmosphere f...


## 6. Testing Different Scenarios

Let's test various types of requests.

In [12]:
# Test scenarios
test_requests = [
    "Turn on the kitchen light",
    "Set up for a party in the living room",
    "I want to focus in my office",
    "Good morning!",  # Scene-based
]

print("Testing Multiple Scenarios")
print("=" * 60)

for request in test_requests:
    print(f"\n📝 Request: \"{request}\"")
    print("-" * 40)
    
    try:
        response, trace = agent.run_with_trace(request)
        print(f"Steps: {len(trace)}")
        print(f"Response: {response[:150]}..." if len(response) > 150 else f"Response: {response}")
    except Exception as e:
        print(f"Error: {e}")

Testing Multiple Scenarios

📝 Request: "Turn on the kitchen light"
----------------------------------------
Steps: 6
Response: I've just turned on the kitchen light for you! If you need anything else, just let me know. 😊

📝 Request: "Set up for a party in the living room"
----------------------------------------
Steps: 6
Response: Let’s get the living room ready for your party! 🎉 I’ll start playing a lively party playlist at a high volume to set the mood, and I’ll also switch th...

📝 Request: "I want to focus in my office"
----------------------------------------
Steps: 6
Response: I’m setting up your office for some serious focus! I’ll turn on the desk lamp to provide bright, cool white light, perfect for concentration. Plus, I’...

📝 Request: "Good morning!"
----------------------------------------
Error: 'NoneType' object has no attribute 'split'


## 7. Understanding the Node Functions

Let's look at what each node does in detail.

In [13]:
# Import individual nodes
from src.agent.nodes import (
    parse_intent,
    retrieve_context,
    check_sufficiency,
    generate_plan,
    validate_plan,
    generate_response,
)

# Create initial state
state = create_initial_state("Make the living room relaxing")

print("Initial State:")
print(f"  user_input: {state['user_input']}")

Initial State:
  user_input: Make the living room relaxing


In [14]:
# Step 1: Parse Intent
print("\n" + "=" * 50)
print("STEP 1: Parse Intent")
print("=" * 50)

result = parse_intent(state)
state.update(result)

print("\nExtracted Intent:")
print(json.dumps(state['parsed_intent'], indent=2))


STEP 1: Parse Intent

Extracted Intent:
{
  "goal": "create a relaxing atmosphere",
  "rooms": [
    "living room"
  ],
  "actions": null,
  "mood": "relaxed",
  "scene_hint": null,
  "constraints": null
}


In [15]:
# Step 2: Retrieve Context
print("\n" + "=" * 50)
print("STEP 2: Retrieve Context (GraphRAG)")
print("=" * 50)

result = retrieve_context(state)
state.update(result)

print(f"\nStrategies used: {state['retrieval_metadata']['strategies_used']}")
print(f"Context length: {len(state['retrieved_context'])} characters")
print("\nContext preview:")
print(state['retrieved_context'][:500])


STEP 2: Retrieve Context (GraphRAG)

Strategies used: ['room_context']
Context length: 875 characters

Context preview:
# Smart Home Context

## Room: Living Room
- Type: entertainment
- Floor: Ground Floor
- Adjacent to: Kitchen

### Devices (6 total)
- **Thermostat** (thermostat) ✓
  - Capabilities: temperature
- **Window Blinds** (blinds) ✓
  - Capabilities: open_close
- **Smart Speaker** (speaker) ✓
  - Capabilities: announce, play_music, volume, power
- **Floor Lamp** (light) ✓
  - Capabilities: dim, power
- **Ceiling Light** (light) ✓
  - Capabilities: color, dim, power
- **Smart TV** (television) ✓
  - Cap


In [16]:
# Step 3: Check Sufficiency
print("\n" + "=" * 50)
print("STEP 3: Check Sufficiency")
print("=" * 50)

result = check_sufficiency(state)
state.update(result)

print(f"\nNeeds clarification: {state.get('needs_clarification', False)}")
if state.get('clarification_question'):
    print(f"Question: {state['clarification_question']}")


STEP 3: Check Sufficiency

Needs clarification: False


In [17]:
# Step 4: Generate Plan (if sufficient)
if not state.get('needs_clarification'):
    print("\n" + "=" * 50)
    print("STEP 4: Generate Plan")
    print("=" * 50)
    
    result = generate_plan(state)
    state.update(result)
    
    print("\nGenerated Action Plan:")
    print(json.dumps(state['action_plan'], indent=2))


STEP 4: Generate Plan

Generated Action Plan:
{
  "reasoning": "To create a relaxing atmosphere, I will dim the lights and close the window blinds to reduce brightness and distractions, while also playing calming music to enhance the mood.",
  "actions": [
    {
      "device_id": "living_blinds_01",
      "device_name": "Window Blinds",
      "capability": "open_close",
      "value": "close",
      "reason": "Closing the blinds helps to create a cozy and private environment."
    },
    {
      "device_id": "living_floor_lamp_01",
      "device_name": "Floor Lamp",
      "capability": "dim",
      "value": 30,
      "reason": "Dimming the floor lamp provides soft lighting, contributing to a relaxed atmosphere."
    },
    {
      "device_id": "living_ceiling_light_01",
      "device_name": "Ceiling Light",
      "capability": "dim",
      "value": 20,
      "reason": "Dimming the ceiling light further enhances the cozy and relaxing environment."
    },
    {
      "device_id": "livi

In [18]:
# Step 5: Validate Plan
if state.get('action_plan'):
    print("\n" + "=" * 50)
    print("STEP 5: Validate Plan")
    print("=" * 50)
    
    result = validate_plan(state)
    state.update(result)
    
    validation = state['validation_result']
    print(f"\nIs valid: {validation['is_valid']}")
    if validation['issues']:
        print(f"Issues: {validation['issues']}")
    if validation['suggestions']:
        print(f"Suggestions: {validation['suggestions']}")


STEP 5: Validate Plan

Is valid: True


In [19]:
# Step 6: Generate Response
if state.get('action_plan'):
    print("\n" + "=" * 50)
    print("STEP 6: Generate Response")
    print("=" * 50)
    
    result = generate_response(state)
    state.update(result)
    
    print(f"\nFinal Response:")
    print(state['final_response'])


STEP 6: Generate Response

Final Response:
I'll create a relaxing atmosphere in the living room for you! I'm closing the blinds to keep things cozy, dimming the floor lamp to 30% and the ceiling light to 20% for soft lighting, and playing some calming music to set the mood. Enjoy your peaceful time! 🌿


## 8. Examining the Prompts

Let's look at the prompts that guide the LLM.

In [20]:
from src.agent.prompts import (
    INTENT_PARSER_PROMPT,
    ACTION_PLAN_PROMPT,
    RESPONSE_PROMPT,
)

print("Intent Parser System Prompt:")
print("=" * 50)
print(INTENT_PARSER_PROMPT.messages[0].prompt.template[:500])

Intent Parser System Prompt:
You are an intent parser for a smart home system.
Your job is to extract structured information from natural language requests.

Extract the following:
1. **goal**: What the user wants to achieve (e.g., "create cozy atmosphere", "prepare for movie")
2. **rooms**: List of rooms mentioned or implied (e.g., ["living room"])
3. **actions**: Specific actions mentioned (e.g., ["dim lights", "turn on TV"])
4. **mood**: Mood/atmosphere if mentioned (e.g., "relaxed", "energetic")
5. **scene_hint**: If a 


In [21]:
print("\nAction Plan System Prompt:")
print("=" * 50)
print(ACTION_PLAN_PROMPT.messages[0].prompt.template[:800])


Action Plan System Prompt:
You are a smart home assistant that creates action plans.
Given the user's intent and available devices, generate a list of specific device commands.

## Available Context
You will receive:
1. User's parsed intent (goal, rooms, mood, etc.)
2. Available devices with their capabilities
3. Relevant scenes (predefined configurations)

## Output Format
Generate a JSON object with:
- **reasoning**: Brief explanation of your choices (1-2 sentences)
- **actions**: List of device commands

Each action must have:
- **device_id**: The device identifier
- **device_name**: Human-readable name
- **capability**: The capability to use (must exist on the device!)
- **value**: The value to set (appropriate for the capability)
- **reason**: Why this action helps achieve the goal

## Important Rules
1. ONLY u


## 9. Exercises

Try these exercises to deepen your understanding.

In [22]:
# Exercise 1: Test a vague request
# Does the agent ask for clarification?

# Your code here:
# response = agent.run("Make it nice")


In [23]:
# Exercise 2: Examine how a scene-based request is handled
# What retrieval strategies are used?

# Your code here:
# state = agent.get_full_state("Movie night mode")
# print(state['retrieval_metadata'])


In [24]:
# Exercise 3: Compare the action plans for similar requests
# Are they consistent?

# Your code here:
# plan1 = agent.get_full_state("Dim the living room lights")['action_plan']
# plan2 = agent.get_full_state("Make the living room darker")['action_plan']


In [25]:
# Exercise 4: Modify a prompt and see the difference
# (Advanced - edit src/agent/prompts.py)

# Ideas:
# - Make the response more formal/casual
# - Add energy-saving considerations
# - Include safety warnings


## 10. Summary

### What We Learned

1. **LangGraph Workflows**
   - Nodes are processing functions
   - State flows through the graph
   - Conditional edges enable branching

2. **Agent Architecture**
   - Parse → Retrieve → Plan → Validate → Respond
   - Each step is modular and testable
   - State accumulates context

3. **GraphRAG Integration**
   - Intent determines retrieval strategy
   - Context is structured for LLM reasoning
   - Multiple strategies can combine

4. **LLM Orchestration**
   - Prompts guide LLM behavior
   - Output parsing structures results
   - Validation ensures quality

### Key Takeaways

- **Separation of concerns**: Each node has one job
- **Explicit state**: Everything is inspectable
- **Flexible routing**: Conditional logic is clear
- **Testable**: Each component can be tested alone

## What's Next?

- Try the CLI: `python app.py`
- Add new scenes to the knowledge graph
- Customize prompts for different styles
- Add new retrieval strategies
- Implement real device control (simulation mode)